# VIBEJOBBER

In [9]:
import requests
import json
import os
from dotenv import load_dotenv
import asyncio
from typing import Annotated

from agents import Agent, Runner, function_tool


load_dotenv(override=True)




True

In [10]:
JOB_BOARD_SITES = (
    "site:lever.co | site:greenhouse.io | site:jobs.ashbyhq.com | "
    "site:app.dover.io | site:workable.com"
)


def build_job_search_query(job_title: str, or_terms: list[str] | None = None) -> str:
    """Build the same style of query as Google: boards + (title | ... OR variants)."""
    title = job_title.strip()
    if not title:
        raise ValueError("job_title must be non-empty")

    variants: list[str] = [title]
    lower = title.lower()
    if "developer" not in lower:
        variants.append(f"{title} developer")
    if "engineer" not in lower:
        variants.append(f"{title} engineer")
    if or_terms:
        for extra in or_terms:
            e = extra.strip()
            if e and e not in variants:
                variants.append(e)

    or_clause = " | ".join(variants)
    return f"{JOB_BOARD_SITES} ({or_clause})"


def search_jobs(
    job_title: str,
    *,
    num: int = 10,
    or_terms: list[str] | None = None,
    hl: str = "en",
    gl: str | None = None,
    past_week: bool = True,
) -> dict:
    """
    Serper Google Search API — job boards only, optional last-week filter (tbs=qdr:w).

    `or_terms` adds extra OR branches inside the parentheses (e.g. ["mobile developer"]).
    """
    api_key = os.getenv("SERPER_API_KEY")
    if not api_key:
        raise RuntimeError("SERPER_API_KEY is not set")

    q = build_job_search_query(job_title, or_terms=or_terms)
    payload: dict = {
        "q": q,
        "num": num,
        "hl": hl,
        "autocorrect": True,
    }
    if gl:
        payload["gl"] = gl
    if past_week:
        payload["tbs"] = "qdr:w"

    resp = requests.post(
        "https://google.serper.dev/search",
        headers={"X-API-KEY": api_key, "Content-Type": "application/json"},
        data=json.dumps(payload),
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()


def search(job_title: str, **kwargs) -> dict:
    """Alias for `search_jobs` (job title → board-scoped Serper query)."""
    return search_jobs(job_title, **kwargs)


In [11]:



@function_tool
def search_job_boards(
    job_title: Annotated[str, "Core role or stack, e.g. Flutter, Staff Backend, React Native"],
    extra_keywords: Annotated[
        str,
        "Optional comma-separated OR terms for the query, e.g. 'mobile developer, iOS'",
    ] = "",
    num_results: Annotated[int, "How many Google results to return (typically 5–10)"] = 10,
    past_week: Annotated[bool, "Restrict to listings Google associates with the past week"] = True,
) -> str:
    """Search Lever, Greenhouse, Ashby, Dover, and Workable via Serper (Google)."""
    extras = [s.strip() for s in extra_keywords.split(",") if s.strip()] or None
    data = search_jobs(
        job_title,
        num=num_results,
        or_terms=extras,
        past_week=past_week,
    )
    organic = data.get("organic") or []
    slim = [
        {"title": o.get("title"), "link": o.get("link"), "snippet": o.get("snippet")}
        for o in organic
    ]
    return json.dumps(
        {"search_query": data.get("searchParameters", {}).get("q"), "results": slim},
        indent=2,
    )


job_hunter_agent = Agent(
    name="Job Hunter",
    instructions=(
        "You help users find job postings on ATS boards: Lever, Greenhouse, Ashby, Dover, and Workable. "
        "When they ask for roles, openings, or who is hiring, call search_job_boards with a clear job_title. "
        "Pass extra_keywords as comma-separated synonyms when the user names related skills (e.g. mobile, iOS with Flutter). "
        "After the tool returns JSON, summarize listings with markdown links [title](url) and short context from snippets. "
        "If results are empty, suggest relaxing extra_keywords or trying a broader job_title."
    ),
    tools=[search_job_boards],
    model="gpt-4o-mini",
)


async def run_job_agent(message: str) -> str:
    """Run the OpenAI agent (async). In Jupyter you can use: await run_job_agent('...')"""
    result = await Runner.run(job_hunter_agent, message)
    return result.final_output

In [12]:
# Direct Serper (no agent):
search_jobs("Flutter", or_terms=["mobile developer"])

# OpenAI agent (needs OPENAI_API_KEY in .env; install: pip install -r requirements.txt):
# print(await run_job_agent("Find Flutter mobile developer jobs posted in the last week"))

# Or from a plain Python script:
# print(asyncio.run(run_job_agent("Who is hiring for senior Python backend roles?")))

{'searchParameters': {'q': 'site:lever.co | site:greenhouse.io | site:jobs.ashbyhq.com | site:app.dover.io | site:workable.com (Flutter | Flutter developer | Flutter engineer | mobile developer)',
  'hl': 'en',
  'type': 'search',
  'num': 10,
  'autocorrect': True,
  'tbs': 'qdr:w',
  'engine': 'google'},
 'organic': [{'title': 'Resilient Co - Flutter Frontend Engineer Sr - Lever',
   'link': 'https://jobs.lever.co/resilientco/82be839f-e643-448d-99b6-f3a22b7bf9dc',
   'snippet': 'We are looking for a Senior Frontend Engineer who will take a key role in driving the architecture and evolution of our Flutter mobile applications.',
   'date': '20 hours ago',
   'position': 1},
  {'title': 'Senior/Staff Flutter Engineer (Identity Defender) - Greenhouse',
   'link': 'http://job-boards.greenhouse.io/expressvpn/jobs/8507836002',
   'snippet': 'As a Senior Flutter Engineer, you will design and build cross-platform applications that power Identity Defender across mobile (iOS, Android), and desk